# Notebook for extracting Learningoutcomes for all schools

## Setup

In [ ]:
from firecrawl_app import app, ExtractSchema, NestedModel
import json
import pandas as pd
from utils.helpers import *

filepath = "config.json"

# Load the JSON file containing config
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)
    
# Get all schools
schools = list(config_data.keys())

print("Schools currently defined in config.json:")
print("-" * 40)
for school in sorted(schools):
    print(f"- {school}") 

## Code for running extraction of learning outcomes for all schools defined in config.json.

In [ ]:

first_write = True 


for school,  school_data in config_data.items():
    
    print(f"\nNow extracting learningoutcomes from {school}")
    
    dfs = []
    
    for study_program, url in school_data['study_programs'].items():
                
        print(f"\n - Extracting data for study program:  {study_program} from {url} ..")
        
        data = app.extract([
                url 
                ], {
                    'prompt': school_data["prompt"],
                    'schema': ExtractSchema.model_json_schema(),                
                })
        
        print(" - Data extracted ..")
        LUBs = data["data"]["læringsutbyttebeskrivelser"]
        df = create_csv(school, study_program, LUBs)
        dfs.append(df)
        
    print(f"\nLearning outcomes for {school} extracted!")
    print(f"\n ----------------------------------------------")
                    
        
    #Concatenating all dataframes
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Append df to file
    filename = "Learning_outcomes.csv"
    
    # Wrting to CSV file
    combined_df.to_csv(filename, mode='a', index=False, header=first_write)
    first_write = False  # Only write header once
    
# Write final CSV to Excel
csv_to_excel(filename)